# Module 1 Coding Assignment: AAPL 10-K Report RAG Chatbot

In this assignment, you will build a simple AI chatbot using Apple's 10-K report and Retrieval Augmented Generation (RAG). We will leverage the power of OpenAI API, LangChain, and FAISS (Facebook AI Similarity Search) to create a RAG chatbot similar to what we've explored in class. This chatbot will be capable of answering questions based on the information contained within the 10-K report.

## Objectives
By completing this assignment, you will achieve the following:
- Gain practical experience in implementing a Retrieval Augmented Generation (RAG) system.
- Learn how to utilize the OpenAI API for embedding generation and chatbot interactions.
- Understand the process of retrieving relevant information from a document using FAISS.
- Develop a functional chatbot capable of providing answers based on a specific knowledge source.

## Instructions
1. Follow the step-by-step guide below to install necessary packages, download data, and build your chatbot.
2. Ensure you have an OpenAI API key and replace placeholders accordingly.
3. Experiment with different queries to test the capabilities of your chatbot.
4. In the summary section at the end, document your observations and answer the provided questions.

Let's begin building your RAG chatbot!



## Step 1: Install Required Packages
- Execute the following code cell to install the necessary packages:

In [1]:
# freezing the coding environment to these specific versions
!pip install sec-edgar-downloader==5.0.3 langchain==0.3.23 langchain_openai==0.3.14 langchain_community==0.3.21 openai==1.72.0 faiss-cpu==1.10.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.9/643.9 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 113.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires lan

## Step 2: Mount Google Drive and Load OpenAI Credential
* Mount your Google Drive to access necessary files.
* Read the OpenAI key and set environment variable


In [2]:
#Mount the google drive
from google.colab import drive
import os
drive.mount('/content/drive')

# You will need to update this path to where you upload your assignment notebooks, API key, and dataset file
pth = '/content/drive/MyDrive/GenAI for Technical Professionals (Hardeep Johar)/Jupyter Notebook/Assignment'
os.chdir(pth)

# Read the key from the file and set the environmental variable
with open("api/openai","r") as f:
  key=f.read().strip()
os.environ["OPENAI_API_KEY"] = key

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# You will need to update this path to where you upload your assignment notebooks, API key, and dataset file
pth = '/content/drive/MyDrive/GenAI for Technical Professionals (Hardeep Johar)/Jupyter Notebook/Assignment'
os.chdir(pth)

# Read the key from the file and set the environmental variable
with open("api/openai","r") as f:
  key=f.read().strip()
os.environ["OPENAI_API_KEY"] = key

##Step 3. Download AAPL 10-K report from SEC Edgar database

[sec_edgar_downloader](https://sec-edgar-downloader.readthedocs.io/en/latest/)

In [ ]:
from sec_edgar_downloader import Downloader

company_name = "ExampleCompany"
email_address = "example@123.com"

# Create the directory if it doesn't exist
os.makedirs("data/sec-edgar-filings", exist_ok=True)

# Create a downloader instance
dl = Downloader(company_name,email_address, download_folder="data/sec-edgar-filings")

# Download the latest 10-K report for Apple Inc.
company_ticker = "AAPL"  # Change this to any other ticker as needed
dl.get("10-K", company_ticker, limit=1, download_details=True)

print("10-K report downloaded. Check the 'data/sec-edgar-filings' folder.")

10-K report downloaded. Check the 'data/sec-edgar-filings' folder.


##Step 4. Parsing with BeautifulSoup

The report we got is in html format and we need to parse it. BeautifulSoup comes in handy.

[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)

In [ ]:
from bs4 import BeautifulSoup

def preprocess_10k_files(download_dir):
    text_data = []
    for root, dirs, files in os.walk(download_dir):
        for file in files:
            if file.endswith(".html"):
                with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                    soup = BeautifulSoup(f, 'html.parser')
                    text_data.append(soup.get_text())
    return text_data

# Load and preprocess 10-K reports
download_dir = "data/sec-edgar-filings"
text_data = preprocess_10k_files(download_dir)


In [ ]:
text_data

['\n\n\n\n\naapl-20240928false2024FY0000320193P1YP1YP1YP1Yhttp://fasb.org/us-gaap/2024#MarketableSecuritiesCurrent http://fasb.org/us-gaap/2024#MarketableSecuritiesNoncurrenthttp://fasb.org/us-gaap/2024#MarketableSecuritiesCurrent http://fasb.org/us-gaap/2024#MarketableSecuritiesNoncurrenthttp://fasb.org/us-gaap/2024#LongTermDebtCurrent http://fasb.org/us-gaap/2024#LongTermDebtNoncurrenthttp://fasb.org/us-gaap/2024#LongTermDebtCurrent http://fasb.org/us-gaap/2024#LongTermDebtNoncurrenthttp://fasb.org/us-gaap/2024#OtherAssetsNoncurrenthttp://fasb.org/us-gaap/2024#OtherAssetsNoncurrenthttp://fasb.org/us-gaap/2024#PropertyPlantAndEquipmentNethttp://fasb.org/us-gaap/2024#PropertyPlantAndEquipmentNethttp://fasb.org/us-gaap/2024#OtherLiabilitiesCurrenthttp://fasb.org/us-gaap/2024#OtherLiabilitiesCurrenthttp://fasb.org/us-gaap/2024#OtherLiabilitiesNoncurrenthttp://fasb.org/us-gaap/2024#OtherLiabilitiesNoncurrenthttp://fasb.org/us-gaap/2024#OtherLiabilitiesCurrenthttp://fasb.org/us-gaap/2024#O

## Step 5: Split the Text for Embedding
- Split the extracted text into smaller chunks for efficient embedding generation.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,  # Adjust as needed
        chunk_overlap=100  # Adjust as needed
    )
    return splitter.split_text(text)

In [ ]:
text_chunks = split_text(text_data[0])

## Step 6: Generate Embeddings
Use a pre-trained model (like OpenAI's text-embedding-ada-002 model or other embedding models) to convert your text data into vector embeddings. Each text entry will be represented as a vector of numbers.

In [ ]:
from langchain_openai import OpenAIEmbeddings

openai_api_key = os.environ.get('OPENAI_API_KEY')
model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=openai_api_key
)

## Step 7: Setup Vectorstore using FAISS(Facebook AI Similarity Search)

In the cell below, we demonstrate how to use the faiss package to create a FAISS Index directly from colab

FAISS (Facebook AI Similarity Search) is a python library for searching and clustering vectors. Recall that we use embedded word vectors to store knowledge in text AI applications. FAISS, with its similarity functionality, helps us retreive knowledge effectively

For more information on Facebook AI Similarity Search, see: https://github.com/facebookresearch/faiss/wiki and for a tutorial on FAISS, see https://github.com/facebookresearch/faiss/wiki/Getting-started

In [ ]:
from langchain_community.vectorstores import FAISS
import faiss

# Step 3: Embed the Text and Store in FAISS
def embed_and_store(text_chunks, embedding_model, index_name):
    faiss_index = FAISS.from_texts(text_chunks, embedding_model)
    faiss_index.save_local(index_name)
    print(f"FAISS index saved at {index_name}")
    return faiss_index


In [ ]:
faiss_index = embed_and_store(text_chunks, embed, "data/aapl_10k_faiss_index")

FAISS index saved at data/aapl_10k_faiss_index


## Step 8: Connect Chatbot to Knowledge Base
* Implement the retrieval function to fetch relevant information from the FAISS index based on user queries.
* We've built a fully-fledged knowledge base. Now it's time to connect that knowledge base to our chatbot.
*To do that, I've prepared the retrieval function that based on the query, retrives the top 3 similar text chunks from the 10-K report we provided earlier.

In [ ]:
import numpy as np

def retrieval(query):
    query_embedding = embed.embed_query(query)
    query_embedding_np = np.array(query_embedding).astype("float32").reshape(1, -1)  # Convert to numpy array
    D, I = faiss_index.index.search(query_embedding_np, k=3) # Retrive the top three similar result
    return [text_chunks[i] for i in I[0]] # Return only the textual content

###Test the retrieval function


In [ ]:
query = "Tell me something about Apple's new products and services?"
retrieved_docs = retrieval(query)
for doc in retrieved_docs:
    print(doc)

Apple Watch Ultra® 2, Apple Watch® Series 10 and Apple Watch SE®. The Company’s line of wireless headphones includes AirPods®, AirPods Pro®, AirPods Max® and Beats® products. Apple Vision Pro™ is the Company’s first spatial computer based on its visionOS™ operating system.Home includes Apple TV®, the Company’s media streaming and gaming device based on its tvOS® operating system, and HomePod® and HomePod mini®, high-fidelity wireless smart speakers.Accessories includes Apple-branded and third-party accessories.Apple Inc. | 2024 Form 10-K | 1ServicesAdvertisingThe Company’s advertising services include third-party licensing arrangements and the Company’s own advertising platforms.AppleCareThe Company offers a portfolio of fee-based service and support products under the AppleCare® brand. The offerings provide priority access to Apple technical support, access to the global Apple authorized service network for repair and replacement services, and in many cases additional coverage for ins

## Step 9: Augment the Prompt
- Create a function to augment user prompts with retrieved information for improved chatbot responses.

In [ ]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = retrieval(query)
    # get the text from the results
    source_knowledge = "\n".join(results)
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query. If the context does not contain enough information for you to answer the query, state it and you might provide general answer based on your knowledge.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

In [ ]:
print(augment_prompt(query))


Using the contexts below, answer the query. If the context does not contain enough information for you to answer the query, state it and you might provide general answer based on your knowledge.

    Contexts:
    Apple Watch Ultra® 2, Apple Watch® Series 10 and Apple Watch SE®. The Company’s line of wireless headphones includes AirPods®, AirPods Pro®, AirPods Max® and Beats® products. Apple Vision Pro™ is the Company’s first spatial computer based on its visionOS™ operating system.Home includes Apple TV®, the Company’s media streaming and gaming device based on its tvOS® operating system, and HomePod® and HomePod mini®, high-fidelity wireless smart speakers.Accessories includes Apple-branded and third-party accessories.Apple Inc. | 2024 Form 10-K | 1ServicesAdvertisingThe Company’s advertising services include third-party licensing arrangements and the Company’s own advertising platforms.AppleCareThe Company offers a portfolio of fee-based service and support products under the AppleC

## Step 10: Test the Chatbot
Test your RAG chatbot with sample queries and observe its responses.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

chat = ChatOpenAI(
    model='gpt-4'
)

messages_RAG = [
    SystemMessage(content="You are a RAG chatbot designed to assist users with information from Apple's 2024 10-K report.")
]

query = "What are the key financial highlights of Apple Inc.?"
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages_RAG.append(prompt)

res_RAG = chat.invoke(messages_RAG)

print(res_RAG.content)

Based on the provided contexts, here are some key financial highlights of Apple Inc. in 2024:

1. Net Sales: Total net sales for Apple Inc. in 2024 were $391,035 million, a 2% increase from 2023. Key contributors to this increase were higher net sales of Services in the Americas and Rest of Asia Pacific, and iPhone sales in Europe and Japan. However, sales in Greater China decreased due to lower net sales of iPhone and iPad.

2. Fiscal Year: Apple's fiscal years 2024 and 2022 spanned 52 weeks each, whereas fiscal year 2023 spanned 53 weeks.

3. Product Announcements: Some significant product introductions during the year 2024 included MacBook Pro 14-in.; MacBook Pro 16-in.; iMac, MacBook Air 13-in.; MacBook Air 15-in.; iPad Air; iPad Pro; software updates for iOS, macOS, iPadOS, watchOS, visionOS, and tvOS; Apple Intelligence™; iPhone 16 versions; Apple Watch Series 10; and AirPods 4.

4. Stock Price Performance: Apple Inc.'s value increased significantly over the years, with its stock

## Step 11: Explore Langchain's RetrievalQA Chain
- Utilize Langchain's `RetrievalQA` chain for a simpler RAG implementation.

In [ ]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# Step 3: Set Up RetrievalQA (RAG Workflow)
def setup_rag(faiss_index):
    print("Setting up RetrievalQA...")
    retriever = faiss_index.as_retriever()
    llm = ChatOpenAI(model="gpt-4o")  # Use GPT-4 for responses
    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type="stuff")
    return qa_chain


In [ ]:
rag_pipeline = setup_rag(faiss_index)


Setting up RetrievalQA...


In [ ]:
rag_pipeline.invoke("What are new products of Apple Inc.?")

{'query': 'What are new products of Apple Inc.?',
 'result': 'Based on the information provided, new product announcements by Apple Inc. during fiscal year 2024 include:\n\n- First Quarter 2024:\n  - MacBook Pro 14-inch\n  - MacBook Pro 16-inch\n  - iMac\n\n- Second Quarter 2024:\n  - MacBook Air 13-inch\n  - MacBook Air 15-inch\n\n- Third Quarter 2024:\n  - iPad Air\n  - iPad Pro\n  - Updates to operating systems: iOS 18, macOS Sequoia, iPadOS 18, watchOS 11, visionOS 2, and tvOS 18\n  - Apple Intelligence™, a personal intelligence system\n\n- Fourth Quarter 2024:\n  - iPhone 16, iPhone 16 Plus, iPhone 16 Pro, and iPhone 16 Pro Max\n  - Apple Watch Series 10\n  - AirPods 4'}

# Summary and Observations
- Reflect on the performance of your RAG chatbot. How effectively did it answer your queries? Were there any limitations or areas where it struggled?
- Discuss the impact of using Retrieval Augmented Generation (RAG) in this scenario. What advantages does RAG offer compared to a traditional chatbot approach?
- Consider potential improvements or extensions for this project. Could you enhance the chatbot's capabilities by incorporating additional data sources or refining the retrieval process?

By completing this assignment, you have gained valuable experience in building a RAG-based chatbot using real-world data. This technology holds immense potential for various applications, such as customer support, information retrieval, and research assistance.

Congratulations on successfully completing this challenging and rewarding project!

